# 🌤️ Notebook 11: Meteorological Extension (Extension 1)

**Objective:** Compare model performance between:
- **Condition A:** Pollutant-only features
- **Condition B:** Pollutant + meteorological features

Each of the 3 architectures (CNN, LSTM, Hybrid) is trained under both conditions.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import load_config, load_csv
from src.preprocessing.scaler import AQIScaler
from src.preprocessing.sequence_generator import create_sequences
from src.models.cnn import build_cnn_model
from src.models.lstm import build_lstm_model
from src.models.cnn_lstm import build_cnn_lstm_model
from src.training.callbacks import get_callbacks
from src.evaluation.metrics import compute_metrics, print_model_metrics
from src.evaluation.model_comparison import compare_pollutant_vs_weather

config = load_config('../config/config.yaml')
df = load_csv('../' + config['data']['featured_path'])

with open('../artifacts/feature_groups.json', 'r') as f:
    feature_groups = json.load(f)

print(f"Data shape: {df.shape}")

## 11.1 Define Feature Sets

In [ ]:
# Pollutant-only features (NO meteorological)
pollutant_only_cols = []
for group in ['base_pollutants', 'lag_features', 'rolling_features', 'cyclic_features', 'ratio_features']:
    if group in feature_groups:
        pollutant_only_cols.extend(feature_groups[group])
pollutant_only_cols = [c for c in pollutant_only_cols if c in df.columns]

# Pollutant + weather features (ALL features)
all_cols = pollutant_only_cols.copy()
if 'meteorological' in feature_groups:
    weather_cols = [c for c in feature_groups['meteorological'] if c in df.columns]
    all_cols.extend(weather_cols)

print(f"Pollutant-only features: {len(pollutant_only_cols)}")
print(f"Pollutant + weather features: {len(all_cols)}")

## 11.2 Helper: Train & Evaluate Under a Condition

In [ ]:
def train_evaluate_condition(df, feature_cols, model_builder, model_name, config):
    """Train a model on given feature set and return metrics."""
    target_col = 'aqi_code'
    window_size = config['sequence']['window_size']
    test_size = config['split']['test_size']
    
    # Scale
    split_idx = int(len(df) * (1 - test_size))
    scaler = AQIScaler()
    scaler.fit(df.iloc[:split_idx], feature_cols)
    df_scaled = scaler.transform(df)
    
    # Create sequences
    X, y = create_sequences(
        df_scaled[feature_cols].values,
        df_scaled[target_col].values.astype(int),
        window_size
    )
    
    split_idx = int(len(X) * (1 - test_size))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    # Build & train
    input_shape = (X_train.shape[1], X_train.shape[2])
    num_classes = len(set(y_train))
    model = model_builder(input_shape, num_classes)
    
    callbacks = get_callbacks(f'meteo_{model_name}', config['training'])
    
    model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=config['training']['epochs'],
        batch_size=config['training']['batch_size'],
        callbacks=callbacks,
        verbose=0
    )
    
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    metrics = compute_metrics(y_test, y_pred)
    
    print(f"  {model_name}: Acc={metrics['accuracy']:.4f}, F1={metrics['f1_score']:.4f}")
    return metrics

## 11.3 Run Comparison

In [ ]:
model_builders = {
    'CNN': lambda s, n: build_cnn_model(s, n, config['models']['cnn']),
    'LSTM': lambda s, n: build_lstm_model(s, n, config['models']['lstm']),
    'Hybrid CNN-LSTM': lambda s, n: build_cnn_lstm_model(s, n, config['models']['cnn_lstm']),
}

results_pollutant = {}
results_weather = {}

for name, builder in model_builders.items():
    print(f"\n--- {name} ---")
    
    print("  Condition A: Pollutant Only")
    results_pollutant[name] = train_evaluate_condition(
        df, pollutant_only_cols, builder, f"{name}_pollutant", config
    )
    
    print("  Condition B: Pollutant + Weather")
    results_weather[name] = train_evaluate_condition(
        df, all_cols, builder, f"{name}_weather", config
    )

## 11.4 Results Comparison

In [ ]:
comparison = compare_pollutant_vs_weather(
    results_pollutant, results_weather,
    save_path='../results'
)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
models = list(model_builders.keys())
x = np.arange(len(models))
w = 0.35

f1_poll = [results_pollutant[m]['f1_score'] for m in models]
f1_wea = [results_weather[m]['f1_score'] for m in models]

bars1 = ax.bar(x - w/2, f1_poll, w, label='Pollutant Only', color='#2196F3')
bars2 = ax.bar(x + w/2, f1_wea, w, label='Pollutant + Weather', color='#FF9800')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=10)

ax.set_ylabel('F1-Score (weighted)')
ax.set_title('Meteorological Extension — Pollutant Only vs Pollutant + Weather')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.savefig('../results/meteorological_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## ✅ Meteorological Extension Complete

**Next:** Notebook 12 — Model Evaluation